In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm, beta as beta_dist, gamma as gamma_dist


In [3]:
def precision_poisson_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    SI: float,
    cl: float,
):

    errors = (
        sample_s.loc[
            (sample_s["E"] != 0),
            ["ER"],
        ]
        .copy()
        .sort_values("ER", ascending=False)
        .reset_index(drop=True)
    )

    taints = errors["ER"].to_numpy(dtype=float)

    basic_rf = gamma_dist.ppf(q=cl, a=1, scale=1)

    BP = SI * basic_rf

    ranks = np.arange(1, len(taints) + 1)

    incremental_factors = (
        gamma_dist.ppf(q=cl, a=ranks + 1, scale=1) - gamma_dist.ppf(q=cl, a=ranks, scale=1) - 1
    )

    IA = SI * np.dot(incremental_factors, taints)

    SE = BP + IA
    ULE = EE + SE
    VAR = 0

    return SE, VAR, ULE

In [ ]:
from scipy.stats import beta as beta_dist
import numpy as np
import pandas as pd


def precision_binomial_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    SI: float,
    cl: float,
):

    errors = (
        sample_s.loc[
            (sample_s["E"] != 0),
            ["ER"],
        ]
        .copy()
        .sort_values("ER", ascending=False)
        .reset_index(drop=True)
    )

    taints = errors["ER"].to_numpy(dtype=float)

    n = len(sample_s)

    basic_rf = n * beta_dist.ppf(q=cl, a=1, b=n,)

    BP = SI * basic_rf

    ranks = np.arange(1, len(taints) + 1)

    incremental_factors = (
        n * (
            beta_dist.ppf(q=cl, a=ranks + 1, b=n - ranks,) - 
            beta_dist.ppf( q=cl, a=ranks, b=n - ranks + 1, )
        )
        - 1
    )
    incremental_factors = np.where(incremental_factors==np.nan, n, incremental_factors)

    IA = SI * np.dot(incremental_factors, taints)

    SE = BP + IA
    ULE = EE + SE
    VAR = 0

    return SE, VAR, ULE

In [4]:
def precision_binomial_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    BV: float,
    cl: float,
    n: int,
):
    errors = (
            sample_s.loc[
                (sample_s["E"] != 0),
                ["ER"],
            ]
            .copy()
            .sort_values("ER", ascending=False)
            .reset_index(drop=True)
        )
    
    taints = errors["ER"].to_numpy(dtype=float)

    BP = (1 - (1 - cl) ** (1 / n))

    ranks = np.arange(1, len(taints) + 1)

    # p_k^U = Beta^-1(cl; k+1, n-k) is undefined at k=n (every sampled item
    # tainted): shape2 = n-k = 0. The analytical limit is 1 exactly, since an
    # error rate cannot exceed 1 -- substitute it there instead of letting it
    # evaluate to NaN. (This can only occur at the single largest rank, and
    # only when every sampled item is tainted; the second term below never
    # hits shape2=0 since n-(k-1) >= 1 for all k in [1, n].)
    b_first = n - ranks
    first_term = np.where(
        b_first == 0,
        1.0,
        beta_dist.ppf(q=cl, a=ranks + 1, b=np.where(b_first == 0, 1, b_first)),
    )
    incremental_factors = first_term - beta_dist.ppf(q=cl, a=ranks, b=n - (ranks-1))

    IA = np.dot(incremental_factors, taints)

    SE = (BP + IA) * BV
    ULE = EE + SE
    VAR = 0

    return SE, VAR, ULE

In [6]:
ranks = np.arange(1, 100 + 1)
incremental_factors = (
        gamma_dist.ppf(q=0.9, a=ranks + 1, scale=1) - gamma_dist.ppf(q=0.9, a=ranks, scale=1) - 1
    )


In [9]:
gamma_dist.ppf(q=0.9, a=ranks, scale=1)

array([  2.30258509,   3.88972017,   5.32232034,   6.68078307,
         7.99358959,   9.27467389,  10.53207211,  11.77091446,
        12.99471154,  14.20599029,  15.40664117,  16.59812214,
        17.78158564,  18.95796127,  20.12801187,  21.29237254,
        22.45157876,  23.60608695,  24.75628991,  25.90252861,
        27.04510123,  28.18427036,  29.32026869,  30.45330351,
        31.5835605 ,  32.71120671,  33.83639308,  34.95925656,
        36.07992185,  37.19850286,  38.315104  ,  39.42982125,
        40.54274306,  41.65395119,  42.76352136,  43.87152387,
        44.97802416,  46.0830832 ,  47.18675798,  48.28910181,
        49.39016466,  50.48999347,  51.58863236,  52.68612289,
        53.78250427,  54.8778135 ,  55.9720856 ,  57.06535369,
        58.15764919,  59.24900191,  60.33944015,  61.42899084,
        62.51767962,  63.60553089,  64.69256795,  65.77881302,
        66.86428734,  67.94901121,  69.03300406,  70.11628448,
        71.1988703 ,  72.28077863,  73.36202585,  74.44

In [19]:
n= 100
cl=0.9
incremental_factors = (
        n * (
            beta_dist.ppf(q=cl, a=ranks + 1, b=n - ranks,) - 
            beta_dist.ppf( q=cl, a=ranks, b=n - ranks + 1, )
        )
        - 1
    )
incremental_factors

array([ 0.55767185,  0.40057927,  0.32404613,  0.27617778,  0.24239403,
        0.21676699,  0.19636771,  0.17955899,  0.16534373,  0.15307484,
        0.14231138,  0.13274091,  0.12413492,  0.11632181,  0.1091698 ,
        0.10257573,  0.0964575 ,  0.09074878,  0.08539534,  0.08035225,
        0.07558198,  0.07105283,  0.0667378 ,  0.06261374,  0.05866062,
        0.05486101,  0.05119966,  0.0476631 ,  0.04423941,  0.04091799,
        0.0376893 ,  0.03454478,  0.03147669,  0.02847798,  0.02554221,
        0.02266348,  0.01983632,  0.01705571,  0.01431693,  0.01161559,
        0.00894754,  0.00630889,  0.00369591,  0.00110506, -0.00146707,
       -0.00402377, -0.00656824, -0.00910359, -0.0116329 , -0.01415918,
       -0.01668545, -0.01921473, -0.02175004, -0.02429444, -0.02685106,
       -0.02942309, -0.03201382, -0.03462665, -0.03726514, -0.03993299,
       -0.04263412, -0.04537265, -0.04815299, -0.05097983, -0.05385822,
       -0.0567936 , -0.05979188, -0.06285949, -0.06600346, -0.06

In [17]:
n= 100
cl=0.9
b_first = n - ranks
first_term = np.where(
    b_first == 0,
    1.0,
    beta_dist.ppf(q=cl, a=ranks + 1, b=np.where(b_first == 0, 1, b_first)),
)
incremental_factors = first_term - beta_dist.ppf(q=cl, a=ranks, b=n - (ranks-1))
first_term*n

array([  3.83394975,   5.23452902,   6.55857515,   7.83475293,
         9.07714696,  10.29391396,  11.49028166,  12.66984066,
        13.83518439,  14.98825923,  16.13057061,  17.26331152,
        18.38744643,  19.50376824,  20.61293804,  21.71551377,
        22.81197127,  23.90272005,  24.98811539,  26.06846764,
        27.14404962,  28.21510245,  29.28184025,  30.34445399,
        31.40311461,  32.45797563,  33.50917528,  34.55683838,
        35.60107779,  36.64199578,  37.67968508,  38.71422986,
        39.74570656,  40.77418454,  41.79972675,  42.82239023,
        43.84222655,  44.85928226,  45.87359919,  46.88521478,
        47.89416233,  48.90047122,  49.90416713,  50.90527219,
        51.90380512,  52.89978135,  53.89321311,  54.88410952,
        55.87247662,  56.85831744,  57.84163199,  58.82241726,
        59.80066722,  60.77637277,  61.74952171,  62.72009862,
        63.6880848 ,  64.65345815,  65.61619301,  66.57626002,
        67.5336259 ,  68.48825325,  69.44010026,  70.38

In [12]:
beta_dist.ppf(q=0.9, a=ranks + 1, b=100-ranks)

array([0.0383395 , 0.05234529, 0.06558575, 0.07834753, 0.09077147,
       0.10293914, 0.11490282, 0.12669841, 0.13835184, 0.14988259,
       0.16130571, 0.17263312, 0.18387446, 0.19503768, 0.20612938,
       0.21715514, 0.22811971, 0.2390272 , 0.24988115, 0.26068468,
       0.2714405 , 0.28215102, 0.2928184 , 0.30344454, 0.31403115,
       0.32457976, 0.33509175, 0.34556838, 0.35601078, 0.36641996,
       0.37679685, 0.3871423 , 0.39745707, 0.40774185, 0.41799727,
       0.4282239 , 0.43842227, 0.44859282, 0.45873599, 0.46885215,
       0.47894162, 0.48900471, 0.49904167, 0.50905272, 0.51903805,
       0.52899781, 0.53893213, 0.5488411 , 0.55872477, 0.56858317,
       0.57841632, 0.58822417, 0.59800667, 0.60776373, 0.61749522,
       0.62720099, 0.63688085, 0.64653458, 0.65616193, 0.6657626 ,
       0.67533626, 0.68488253, 0.694401  , 0.7038912 , 0.71335262,
       0.72278469, 0.73218677, 0.74155817, 0.75089814, 0.76020582,
       0.7694803 , 0.77872055, 0.78792544, 0.79709372, 0.80622